# 07_02 An Iliad model: the same code, a new poem

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `data/homer/iliad/greek/`
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/model.py`
>
> **This notebook:** Is our code really general? We run the same steps on the Iliad, change only the folder, and find the one place that needs a small fix.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to point the same code at a new poem: two settings, `BOOKS_DIR` and `BOOKS`
- How to build the Iliad's tokenizer, split 90/10 and train briefly
- How to read the train and validation losses, and generate lines
- The exact code to train a full Iliad model yourself, and how long it takes

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **glob pattern** | a file-name template where `*` stands for anything: `book_*.txt` matches `book_01.txt`, `book_02.txt`, … | `folder.glob("book_*.txt")` |
| **slice (of text)** | only the first part of a text, so a training run takes seconds instead of hours | the first 10,000 characters of Book 1 |

## 3. The idea

### ❓ Do I need new code for the Iliad?

Almost none. The steps are the ones you know, and only the folder changes:

```
folder ─▶ books ─▶ tokenizer ─▶ numbers ─▶ split 90/10 ─▶ train ─▶ check ─▶ write
(new)     02_02     02_01       02_04      05_05          05_01    05_05    06_02
```

Two settings in `train_homer.py` pick the text. `BOOKS_DIR` says which poem (a folder in `data/homer/`), and `BOOKS` says which books (file names). `load_books(folder)` takes the folder as an argument, which is why a new folder works at once ([A_07](../A_toolbox/A_07_organising_code.ipynb)). Let's point it at the Iliad's Greek folder.

👉 **Predict first:** `BOOKS` names only `book_01.txt`. How many books will `load_books` return for the Iliad folder? And for the Odyssey folder?

In [ ]:
from homer_ai.lessons import HOMER  # the project's data/homer folder
from homer_ai.train_homer import BOOKS, BOOKS_DIR, load_books

iliad_folder = HOMER / "iliad" / "greek"

print("BOOKS setting: ", BOOKS)
print("Odyssey folder:", len(load_books(BOOKS_DIR)), "book loaded")
print("Iliad folder:  ", len(load_books(iliad_folder)), "book loaded")
iliad_files = sorted(iliad_folder.glob("book_*.txt"))
print("Iliad book files:", len(iliad_files), [f.name for f in iliad_files[:3]])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

You get 1 book from each folder, because `BOOKS = ["book_01.txt"]` asks for Book 1 only. The Iliad folder has 24 book files, found with a **glob pattern**, a file-name template where `*` stands for anything. It's the same code and the same file names (`book_01.txt` …), so changing the poem really is just a change of folder.

</details>

### ❓ How do I load all 24 books?

Pass the file names yourself: `load_books(folder, books=[...])`. In `train_homer.py` you'd widen the `BOOKS` setting the same way.

👉 **Predict first:** how many books, and how long is Book 1?

In [ ]:
all_names = [f.name for f in iliad_files]  # book_01.txt ... book_24.txt
iliad = load_books(iliad_folder, books=all_names)
print(len(iliad), "books; book 1 has", len(iliad[0]), "characters")
print("first verse:", iliad[0].splitlines()[0])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

That's 24 books, and the first verse is the opening of the Iliad that we met in [07_01](07_01_the_corpus.ipynb). From here on, the steps are exactly the ones from the earlier chapters.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how do I build the Iliad's tokenizer?

Glue the 24 books together and give the text to `CharTokenizer` ([02_01](../02_text_to_numbers/02_01_tokens_and_vocabulary.ipynb), [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb)). Then we do the usual check: encode a verse, decode it again, and see if we get the same text back.

👉 **Predict first:** the Iliad's vocabulary was 171 in [07_00](07_00_chapter_overview.ipynb). What number will `μῆνιν` (5 characters) become: 5 numbers, or fewer?

In [ ]:
import torch

from homer_ai.tokenizer import CharTokenizer

text = "\n".join(iliad)
tokenizer = CharTokenizer(text)
print("vocabulary size:", tokenizer.vocab_size)

first_verse = iliad[0].splitlines()[0]
ids = tokenizer.encode(first_verse)
print("μῆνιν →", tokenizer.encode("μῆνιν"))
print("round trip works:", tokenizer.decode(ids) == first_verse)
print("<|unk|> =", tokenizer.unk_id, "  <|endoftext|> =", tokenizer.eot_id)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

171 tokens (169 characters and the two special ones), `μῆνιν` becomes 5 numbers, and the round trip is exact. The special tokens sit at the end, 169 and 170, as always.

</details>

### ❓ Your turn: how many characters are new in the Iliad?

The Iliad's vocabulary is bigger than the 137 of Odyssey Book 1. Count the different characters the Iliad uses that Book 1 doesn't have. (`text` is the whole Iliad.)

👉 **Predict first:** what would Book 1's tokenizer make of such a character?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
odyssey_book1 = load_books(BOOKS_DIR)[0]

# 👉 do: the number of different characters that are in `text` but not in odyssey_book1
# 💡 hint: set(a) - set(b) keeps what is in a and not in b
new_characters = None

In [ ]:
from homer_ai.lessons import check

check(
    "characters only the Iliad has",
    new_characters,
    len({ch for ch in text if ch not in odyssey_book1}),
    coach="Got 1? That's the other way round: what Book 1 has and the Iliad "
    "doesn't. Got 34? Subtracting the two sizes isn't enough, because Book 1 has "
    "a character of its own. A set and not a number? Wrap it in len().",
)

<details><summary>💡 Hint</summary>

Make a set of each text's characters, take the difference, and count it with `len`.

</details>

<details><summary>✅ Answer</summary>

```python
odyssey_book1 = load_books(BOOKS_DIR)[0]

new_characters = len(set(text) - set(odyssey_book1))
```

35. Book 1's tokenizer would turn every one of them into `<|unk|>`, so a model trained with that vocabulary couldn't even read the Iliad properly. That's why this notebook builds a new tokenizer, and why [07_03](07_03_comparing_fairly.ipynb) gives both poems one shared vocabulary before it compares anything.

</details>

### ❓ Step 2: which text do I train on, and how do I split it?

The whole Iliad is 690,573 characters, which is too long for a notebook. So we use a **slice**, the first 10,000 characters of Book 1 (Book 1 alone is 27,201). Then comes the split of [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb): the first 90% to train on, the last 10% hidden for the check.

👉 **Predict first:** how many numbers go to training and how many to validation?

In [ ]:
SLICE = 10000
data = torch.tensor(tokenizer.encode(iliad[0][:SLICE]))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("training:  ", len(train_data))
print("validation:", len(val_data))
print("validation text starts:", repr(tokenizer.decode(val_data[:30].tolist())))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

You get 9,000 and 1,000, and the held-out text begins in the middle of a verse. The model never trains on those 1,000 characters.

</details>

### ❓ Step 3: how good is the untrained model, and how do I train it?

The measure is `evaluate` from [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb) (the loss without learning). An untrained model guesses among 171 tokens, so its loss should be near `ln(171)`.

Then we run the training loop of [05_01](../05_training_on_the_odyssey/05_01_loss_and_epochs.ipynb) on `train_data` only, with a fixed seed ([05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb)), for 6 epochs. It prints the train and validation loss after each one. We also time it.

👉 **Predict first:** will the validation loss end near the training loss, or far above?

In [ ]:
import math
import time

import torch.nn.functional as F

from homer_ai.lessons import CHUNK, evaluate  # evaluate: the function of 05_05
from homer_ai.model import HomerGreekLLM

torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
untrained = evaluate(model, val_data)
print(f"untrained validation loss: {untrained:.3f}   (ln(171) = {math.log(171):.3f})")

train_curve, val_curve, steps = [], [], 0
started = time.perf_counter()
for epoch in range(6):
    losses = []
    for i in range(0, len(train_data) - CHUNK, CHUNK):
        inputs = train_data[i : i + CHUNK]
        targets = train_data[i + 1 : i + CHUNK + 1]
        loss = F.cross_entropy(model(inputs), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        steps += 1
    train_curve.append(sum(losses) / len(losses))
    val_curve.append(evaluate(model, val_data))
    print(
        f"epoch {epoch + 1} | train {train_curve[-1]:.3f} "
        f"| validation {val_curve[-1]:.3f}"
    )
seconds = time.perf_counter() - started
print(f"{steps} steps in {seconds:.1f} seconds")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The untrained loss is 5.165, right at `ln(171) = 5.142` (pure guessing). After six epochs the training loss is 2.805 and the validation loss 2.861. They're close together, so the model learned something real about Greek letters and didn't just memorise. The validation loss stopped improving at epoch 5 (2.855, then 2.861), a small sign that this slice has little left to teach. The graph shows it.

</details>

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, GOOD, studio_axes, studio_figure

epochs = list(range(1, 7))
title = "Training and validation loss, 6 epochs"
fig = studio_figure("the Iliad model", title, (9, 5.5))
ax = studio_axes(fig, (0.1, 0.14, 0.8, 0.64))
style = {"marker": "o", "linewidth": 2.5}
ax.plot(epochs, train_curve, color=GOOD, label="training loss", **style)
ax.plot(epochs, val_curve, color=ACCENT, label="validation loss", **style)
for curve, colour, shift in ((val_curve, ACCENT, 6), (train_curve, GOOD, -16)):
    ax.annotate(
        f"{curve[-1]:.2f}", (6, curve[-1]), textcoords="offset points",
        xytext=(8, shift), color=colour, fontsize=11,
        fontfamily="monospace", fontweight="bold",
    )
ax.set_xlim(0.7, 6.9)
ax.set_xlabel("epoch")
ax.set_ylabel("average loss")
ax.legend(frameon=False, labelcolor="white")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Both lines fall together and stay close. That's learning, not memorising ([05_05](../05_training_on_the_odyssey/05_05_validation.ipynb)). And this was only six epochs on a small slice, so there's more to gain with more text and more epochs.

</details>

### ❓ Step 4: what does the Iliad model write?

We try the same two ways as [06_02](../06_making_it_talk/06_02_less_repetition.ipynb): greedy (always the best letter, `top_k=1`) and top-k with a little temperature. The prompt is the Iliad's first word, `μῆνιν`. A seed makes it repeatable, and `⏎` is a line break.

👉 **Predict first:** which will loop, and will either write real Greek sentences?

In [ ]:
from homer_ai.lessons import write  # the function of 06_02; top_k=1 is greedy

print("greedy:", write(model, tokenizer, "μῆνιν", 70, top_k=1))
print("top-5: ", write(model, tokenizer, "μῆνιν", 70, temperature=0.8, top_k=5))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Greedy loops at once, on `δʼ` (that's `δέ`, "but", with its last vowel dropped, and the mark `ʼ` shows the drop). Top-5 writes short Greek-looking pieces and a few real small words (`μέν`, `ἐγὼ` "I", `δὲ`), in no sensible order. That's what a few seconds of training on 9,000 characters buys you: the shape of Greek words, but no Greek sentences.

</details>

### ❓ How do I train a FULL Iliad model myself?

With the same loop, on all 24 books of the Iliad folder. It saves to a **new** file, so the Odyssey's `homer_model.pt` is never overwritten. Save this as `train_iliad.py` in the project folder. It needs only functions from `train_homer.py`. `train` prints a line at every 50th epoch, so this short run prints just one, at the end:

```python
# train_iliad.py: a full Iliad model. Save it in the project folder, then run:
#     uv run python train_iliad.py
from pathlib import Path

import torch

from homer_ai.model import HomerGreekLLM
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import save_checkpoint, train

ILIAD = Path("data/homer/iliad/greek")
files = sorted(ILIAD.glob("book_*.txt"))
books = [f.read_text(encoding="utf-8") for f in files]
text = "\n".join(books)  # all 24 books: to build the vocabulary
tokenizer = CharTokenizer(text)

ids = []  # the numbers, with <|endoftext|> between books (the loop of load_data)
for book in books:
    if ids:
        ids.append(tokenizer.eot_id)
    ids.extend(tokenizer.encode(book))
data = torch.tensor(ids, dtype=torch.long)

torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
train(model, optimizer, data, epochs=50)  # prints one line, at epoch 50

# the notes saved with the model: "poem" and "books" tell the MCP server which
# text to search for real lines (without them it would search Odyssey Book 1)
notes = {
    "seed": 42,
    "epochs": 50,
    "lr": 0.01,
    "chunk": 24,
    "train_chars": len(data),
    "val_chars": 0,  # trained on all 24 books: no held-out text
    "books": [f.name for f in files],
    "poem": "iliad",
}
save_checkpoint(model, text, Path("iliad_model.pt"), notes)  # NOT homer_model.pt
```

How long will it take? The code below estimates it from the speed we just measured in this notebook.

👉 **Predict first:** one epoch of the full Iliad is about 28,800 chunks of 24 characters, and we just did a few thousand steps in a few seconds. About how long is one epoch?

In [ ]:
seconds_per_step = seconds / steps
steps_per_epoch = len(text) // CHUNK  # about: one step per 24-character chunk
one_epoch = seconds_per_step * steps_per_epoch
print(f"one step:   {seconds_per_step * 1000:.1f} milliseconds")
print(f"one epoch:  about {one_epoch:.0f} seconds ({steps_per_epoch:,} steps)")
print(f"50 epochs:  about {50 * one_epoch / 60:.0f} minutes")
print(f"800 epochs: about {800 * one_epoch / 3600:.0f} hours (the Odyssey's setting)")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About a millisecond per step, so one epoch of the full Iliad takes about half a minute on this machine (your number will differ), 50 epochs about 25 minutes, and the 800 epochs of the Odyssey model about 6 hours. Start with `epochs=50`, and let it run while you do something else. The file `iliad_model.pt` can be loaded just like in [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb). The MCP server doesn't look for `iliad_model.pt` by itself, so it keeps offering the Odyssey model unless you point `HOMER_MODEL` at the new file ([06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb)).

And the `notes` at the end of the script matter there. The server shows real lines that contain the seed, and it finds them by reading the books named in the checkpoint: `poem` picks the folder in `data/homer/`, `books` the files (`training_text` in `checkpoint.py`). Leave the notes out and it falls back to Odyssey Book 1, so an Iliad model would answer with Odyssey lines. One thing the notes can't fix: the tool's description still says "Homer's Odyssey, Book 1". If you serve an Iliad model for real, change that docstring too, or it won't be true.

</details>

### ❓ Why didn't this notebook save a model?

Because training here is a demo. Saving would make a file nobody asked for, and we never write `homer_model.pt` from a notebook. The script above is the real thing, and you run it on purpose.

## 5. Recap

- The Iliad needs the same code on a different folder. `BOOKS_DIR` picks the poem and `BOOKS` picks the books (`load_books(folder, books=[...])`).
- A short run on a slice gives a loss near 3 and no overfitting. The writing is word-shaped pieces, not Greek sentences.
- A full run is `train_iliad.py` above. It takes about half a minute per epoch on this machine and saves to `iliad_model.pt`.

**You can now:**

- load any poem's books with `load_books(folder, books=[...])`
- find the characters one text has and another lacks
- train and check a small model on a new text with the code you already have

**Next:** [07_03: Comparing fairly](07_03_comparing_fairly.ipynb).